In [1]:
spark

In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
#user for higher level api
spark=SparkSession.builder \
.appName('WordCount') \
.master('yarn')\
.getOrCreate()


25/12/03 18:49:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
data=['Goku karthik manasa','karthik Goku Karthik','manasa karthik pranavi vegetha']
data

['Goku karthik manasa',
 'karthik Goku Karthik',
 'manasa karthik pranavi vegetha']

In [6]:
#for lower level api we used sparkContext.parallelization(data)

rdd=spark.sparkContext.parallelize(data)



In [7]:
rdd.collect()+

['Goku karthik manasa',
 'karthik Goku Karthik',
 'manasa karthik pranavi vegetha']

In [26]:
# Import the  files from hdfs file system to spark as a rdd
rdd_hdfs=spark.sparkContext.textFile('/tmp/input_spark.txt')
rdd_hdfs.collect()


['Goku karthik manasa',
 'karthik Goku Karthik',
 'manasa karthik pranavi vegetha']

In [27]:
rdd_hdfs_mapped=rdd_hdfs.map(lambda x:x.split(' '))
print(rdd_hdfs_mapped.collect())

[['Goku', 'karthik', 'manasa'], ['karthik', 'Goku', 'Karthik'], ['manasa', 'karthik', 'pranavi', 'vegetha']]


In [33]:
rdd_hdfs_mapped_processed = rdd_hdfs_mapped.map(
    lambda lst: [word.strip() for word in lst if word.strip() and word.strip() != ',']
)
rdd_hdfs_mapped_processed.collect()

[['Goku', 'karthik', 'manasa'],
 ['karthik', 'Goku', 'Karthik'],
 ['manasa', 'karthik', 'pranavi', 'vegetha']]

In [40]:
rdd_filtered_hdfs=rdd_hdfs_mapped_processed.filter(lambda x:"manasa" in x)
rdd_filtered_hdfs.collect()

[['Goku', 'karthik', 'manasa'], ['manasa', 'karthik', 'pranavi', 'vegetha']]

In [41]:
words = rdd_filtered_hdfs.flatMap(lambda x: x)
print(words.collect()) 


['Goku', 'karthik', 'manasa', 'manasa', 'karthik', 'pranavi', 'vegetha']


In [42]:
word_map=words.map(lambda word:(word,1))
word_map.collect()

[('Goku', 1),
 ('karthik', 1),
 ('manasa', 1),
 ('manasa', 1),
 ('karthik', 1),
 ('pranavi', 1),
 ('vegetha', 1)]

In [43]:
result=word_map.reduceByKey(lambda a,b:a+b)
result.collect()

[('Goku', 1), ('pranavi', 1), ('vegetha', 1), ('karthik', 2), ('manasa', 2)]